# Explorar `rt/lowstate`

Carga una grabación hecha con `scripts/02_record_lowstate.py` y mira las señales del robot.

**Para generar datos** (dos terminales WSL, con el simulador ya corriendo):
```bash
python scripts/02_record_lowstate.py --dur 13   # terminal A
python scripts/01_stand.py                      # terminal B (para que haya movimiento)
```

**Kernel**: usar el venv `~/.venvs/go2` de WSL (en VSCode: seleccionar kernel → Python de WSL).

Recordatorio del orden de los 12 motores: `FR_hip, FR_thigh, FR_calf, FL_…, RR_…, RL_…`.
En cada pata: **hip** = abducción (abre/cierra lateral), **thigh** = muslo, **calf** = rodilla.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

rec_file = sorted(Path("../data").glob("lowstate_*.npz"))[-1]  # el más reciente
d = np.load(rec_file)
print(rec_file.name)
for k in d.files:
    print(f"  {k:6s} {d[k].shape}")
hz = len(d['t']) / (d['t'][-1] - d['t'][0])
print(f"~{hz:.0f} Hz durante {d['t'][-1]:.1f} s")

LEGS = ['FR', 'FL', 'RR', 'RL']
JOINTS = ['hip', 'thigh', 'calf']

## Posiciones articulares durante el stand

Si la grabación fue durante `01_stand.py` deberías ver la rampa suave (tanh): el muslo baja de ~1.22 a ~0.61 rad y la rodilla sube de ~-2.44 a ~-1.22 al pararse, y vuelven al agacharse.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharex=True)
for j, ax in enumerate(axes):
    for leg in range(4):
        ax.plot(d['t'], d['q'][:, leg * 3 + j], label=LEGS[leg])
    ax.set_title(f'q {JOINTS[j]} [rad]')
    ax.set_xlabel('t [s]')
axes[0].legend()
fig.tight_layout()

## IMU: orientación y velocidad angular

La IMU es el sentido del equilibrio: roll/pitch/yaw del cuerpo y el giróscopo (velocidad angular). Parándose derecho, roll y pitch deberían quedarse cerca de 0; el giróscopo muestra las sacudidas de la transición.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.5), sharex=True)
for i, name in enumerate(['roll', 'pitch', 'yaw']):
    a1.plot(d['t'], d['rpy'][:, i], label=name)
    a2.plot(d['t'], d['gyro'][:, i], label=name)
a1.set_title('orientación [rad]'); a1.legend(); a1.set_xlabel('t [s]')
a2.set_title('giróscopo [rad/s]'); a2.set_xlabel('t [s]')
fig.tight_layout()

## Torques y contactos de pie

`tau_est` es el torque que cada motor está ejerciendo (acá, la salida del control PD). `foot` son los sensores de contacto de los pies — **ojo: el bridge Python de unitree_mujoco no los llena nunca (quedan en 0)**; el robot real y la versión C++ del sim sí los publican. Es un buen ejemplo de que hay que verificar qué simula tu simulador y qué no.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.5), sharex=True)
for j in range(3):
    a1.plot(d['t'], d['tau'][:, j], label=f'FR_{JOINTS[j]}')
for leg in range(4):
    a2.plot(d['t'], d['foot'][:, leg], label=LEGS[leg])
a1.set_title('tau_est pata FR [N·m]'); a1.legend(); a1.set_xlabel('t [s]')
a2.set_title('fuerza de pie [u.a.]'); a2.legend(); a2.set_xlabel('t [s]')
fig.tight_layout()

## Para seguir jugando

- Graficar `dq` (velocidades): ¿qué articulación se mueve más rápido en la transición?
- Derivar `q` numéricamente (`np.gradient(q, t, axis=0)`) y comparar con `dq` — ¿coinciden? ¿por qué habría diferencias en el robot real?
- Grabar una corrida donde empujás al robot en el viewer de MuJoCo (Ctrl+click derecho arrastrando aplica fuerzas) y mirar cómo reaccionan IMU y torques.
- Comparar el `hz` efectivo del sim Python (~200 Hz) con el del robot real (~500 Hz).